### Source 1: SEC EDGAR (10-K filings)

In [ ]:
!pip install sec-edgar-downloader

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 5.2 MB/s eta 0:00:00


In [ ]:
from sec_edgar_downloader import Downloader
import os
import shutil

tenk_path = "/content/drive/My Drive/ESG_proj/10k_txt"
os.makedirs(tenk_path, exist_ok=True)

dl = Downloader("LN", "likhn02@gmail.com")
tickers = ["MSFT", "AAPL", "XOM", "CVX", "NEE", "WMT", "JPM", "PG"]

for ticker in tickers:
    dl.get("10-K", ticker, after="2023-01-01", before="2024-01-01")

# Copy to Drive instead of cwd
for root, dirs, files in os.walk("./sec-edgar-filings"):
    for file in files:
        if file == "full-submission.txt":
            ticker = root.split("/")[2]
            src = os.path.join(root, file)
            dst = os.path.join(tenk_path, f"{ticker}_10K_2023.txt")
            shutil.copy(src, dst)
            print(f"Saved to: {dst}")

Saved to: /content/drive/My Drive/ESG_proj/10k_txt/WMT_10K_2023.txt
Saved to: /content/drive/My Drive/ESG_proj/10k_txt/MSFT_10K_2023.txt
Saved to: /content/drive/My Drive/ESG_proj/10k_txt/PG_10K_2023.txt
Saved to: /content/drive/My Drive/ESG_proj/10k_txt/NEE_10K_2023.txt
Saved to: /content/drive/My Drive/ESG_proj/10k_txt/CVX_10K_2023.txt
Saved to: /content/drive/My Drive/ESG_proj/10k_txt/AAPL_10K_2023.txt
Saved to: /content/drive/My Drive/ESG_proj/10k_txt/JPM_10K_2023.txt


### Source 2: ESG Reports PDFs

#### Microsoft, apple, JPMorgan, Walmart, ExxonMobil, Chevron, NextEra Energy, Proctor & Gamble

In [ ]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 112.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import os

# Ensure a clean mountpoint by removing existing directory if it contains files
if os.path.exists('/content/drive') and os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    # If it contains files, delete the directory to allow a clean mount
    import shutil
    shutil.rmtree('/content/drive')

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import fitz
import os

base_path = "/content/drive/My Drive/ESG_proj/2022_ESG_reports"

tickers = ["MSFT", "AAPL", "JPM", "WMT", "XOM", "CVX", "NEE", "PG"]

for ticker in tickers:
    pdf_path = os.path.join(base_path, f"{ticker}_esg_report.pdf")
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()

    txt_path = os.path.join(base_path, f"{ticker}_SR_2022.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"Saved: {txt_path}")

Saved: /content/drive/My Drive/ESG_proj/2022_ESG_reports/MSFT_SR_2022.txt
Saved: /content/drive/My Drive/ESG_proj/2022_ESG_reports/AAPL_SR_2022.txt
Saved: /content/drive/My Drive/ESG_proj/2022_ESG_reports/JPM_SR_2022.txt
Saved: /content/drive/My Drive/ESG_proj/2022_ESG_reports/WMT_SR_2022.txt
Saved: /content/drive/My Drive/ESG_proj/2022_ESG_reports/XOM_SR_2022.txt
Saved: /content/drive/My Drive/ESG_proj/2022_ESG_reports/CVX_SR_2022.txt
Saved: /content/drive/My Drive/ESG_proj/2022_ESG_reports/NEE_SR_2022.txt
Saved: /content/drive/My Drive/ESG_proj/2022_ESG_reports/PG_SR_2022.txt


### Forward-looking emissions claims Extraction from 2022 ESG Reports

- things like "we will reduce Scope 2 by 30% by 2030".

In [ ]:
# =============================================================
# EXTRACTOR
# Pipeline: spaCy sentence split  ->  regex pre-filter  ->  zero-shot classifier
# =============================================================

import os
import re
import spacy
import pandas as pd
from transformers import pipeline

nlp = spacy.load("en_core_web_sm")

# zero shot classifier
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0,
)

candidate_labels = [
    "forward-looking emissions commitment",
    "backward-looking emissions report",
    "general sustainability statement",
    "unrelated",
]


# regex pre-filter
# A sentence must contain ALL THREE feature types to pass.

# Future-pointing modal verbs / commitment language
MODAL_PATTERNS = [
    r"\bwill\b",
    r"\bplan(s|ned|ning)?\b",
    r"\bcommit(s|ted|ment|ting)?\b",
    r"\btarget(s|ed|ing)?\b",
    r"\baim(s|ed|ing)?\b",
    r"\bintend(s|ed|ing)?\b",
    r"\bgoal\b",
    r"\bpledge(s|d)?\b",
    r"\bby\s+20[2-9]\d\b",
    r"\bby\s+the\s+end\s+of\s+20[2-9]\d\b",
]

# Numbers, percentages, or future years (>= 2023)
NUMBER_PATTERNS = [
    r"\b\d+(\.\d+)?\s*(percent|%)\b",
    r"\b\d+(\.\d+)?\s*(million|billion)\b",
    r"\b20(2[3-9]|[3-9]\d)\b",
    r"\bnet[\s\-]?zero\b",
    r"\bcarbon[\s\-]?(negative|neutral)\b",
    r"\bzero[\s\-]?(waste|emission(s)?)\b",
    r"\b100\s*(percent|%)\b",
]

# Emissions / energy / environmental scope
SCOPE_PATTERNS = [
    r"\bemission(s)?\b",
    r"\bcarbon\b",
    r"\bscope\s*[123]\b",
    r"\bghg\b",
    r"\bgreenhouse\s+gas(es)?\b",
    r"\brenewable(s)?\b",
    r"\bclean\s+energy\b",
    r"\belectricity\b",
    r"\bfossil\s+fuel(s)?\b",
    r"\bdecarboniz(e|ation|ing)\b",
    r"\bclimate\b",
]

modal_re  = re.compile("|".join(MODAL_PATTERNS),  flags=re.IGNORECASE)
number_re = re.compile("|".join(NUMBER_PATTERNS), flags=re.IGNORECASE)
scope_re  = re.compile("|".join(SCOPE_PATTERNS),  flags=re.IGNORECASE)


def feature_check(sentence: str) -> dict:
    """Return which feature classes the sentence contains."""
    return {
        "has_modal":  bool(modal_re.search(sentence)),
        "has_number": bool(number_re.search(sentence)),
        "has_scope":  bool(scope_re.search(sentence)),
    }


def passes_prefilter(sentence: str) -> bool:
    f = feature_check(sentence)
    return f["has_modal"] and f["has_number"] and f["has_scope"]


# main extraction loop

base_path = "/content/drive/My Drive/ESG_proj/esg_txt"
output_path = "/content/drive/My Drive/ESG_proj/extracted_claims_v2"
os.makedirs(output_path, exist_ok=True)

tickers = ["MSFT", "AAPL", "JPM", "WMT", "CVX", "XOM", "NEE", "PG"]

all_rows = []

for ticker in tickers:
    txt_path = os.path.join(base_path, f"{ticker}_SR_2022.txt")
    with open(txt_path, "r", encoding="utf-8") as f:
        text = f.read()

    # Clean up PDF artifacts: collapse whitespace, fix mid-word line breaks
    text = re.sub(r"-\n", "", text)
    text = re.sub(r"\s+", " ", text)

    # spaCy sentence split (handles "U.S.", "Scope 1.", decimals, etc.)
    doc = nlp(text)
    sentences = [
        sent.text.strip()
        for sent in doc.sents
        if 30 <= len(sent.text.strip()) <= 500   # drop tiny fragments + huge chunks
    ]

    # regex pre-filter
    candidates = []
    for sent in sentences:
        feats = feature_check(sent)
        if feats["has_modal"] and feats["has_number"] and feats["has_scope"]:
            candidates.append((sent, feats))

    print(f"{ticker}: {len(sentences)} sentences -> {len(candidates)} pre-filtered")

    # zero-shot classifier (only on what survived)
    rows = []
    for sent, feats in candidates:
        result = classifier(sent, candidate_labels)
        top_label = result["labels"][0]
        top_score = result["scores"][0]

        rows.append({
            "ticker": ticker,
            "sentence": sent,
            "has_modal": feats["has_modal"],
            "has_number": feats["has_number"],
            "has_scope": feats["has_scope"],
            "top_label": top_label,
            "top_score": round(top_score, 3),
            "is_forward_commitment": (
                top_label == "forward-looking emissions commitment"
                and top_score > 0.5
            ),
        })

    df_company = pd.DataFrame(rows)
    out_csv = os.path.join(output_path, f"{ticker}_claims_v2.csv")
    df_company.to_csv(out_csv, index=False)

    n_kept = df_company["is_forward_commitment"].sum() if len(df_company) else 0
    print(f"  -> classifier kept {n_kept} as forward-commitment "
          f"(saved {len(df_company)} candidates total)")

    all_rows.extend(rows)

# Combined CSV across all companies
df_all = pd.DataFrame(all_rows)
df_all.to_csv(os.path.join(output_path, "all_claims_v2.csv"), index=False)
print(f"\nDONE. Total candidates across 8 companies: {len(df_all)}")
print(f"Of these, classified as forward-commitment: "
      f"{df_all['is_forward_commitment'].sum()}")


'''# quick before/after summary table

# Compare v1 (your original output) counts vs v2 candidate + classifier counts.
# Adjust v1_counts if you have the exact numbers handy.
summary = (
    df_all.groupby("ticker")
          .agg(
              v2_candidates=("sentence", "count"),
              v2_forward_commitments=("is_forward_commitment", "sum"),
              avg_score=("top_score", "mean"),
          )
          .round(3)
)
print(summary) '''


# "parseability" eval
# Try to extract structured fields. Fraction with >=2 fields is a quality signal.

YEAR_RE     = re.compile(r"\bby\s+(20\d{2})\b", re.IGNORECASE)
PERCENT_RE  = re.compile(r"(\d+(?:\.\d+)?)\s*(?:percent|%)", re.IGNORECASE)
SCOPE_NUM   = re.compile(r"\bscope\s*([123])\b", re.IGNORECASE)
BASELINE_RE = re.compile(r"\b(?:from\s+(?:a\s+)?)?(20\d{2})\s+baseline\b", re.IGNORECASE)


def parse_claim(sentence: str) -> dict:
    return {
        "target_year":  (m.group(1) if (m := YEAR_RE.search(sentence)) else None),
        "target_value": (m.group(1) + "%" if (m := PERCENT_RE.search(sentence)) else None),
        "scope":        (m.group(1) if (m := SCOPE_NUM.search(sentence)) else None),
        "baseline":     (m.group(1) if (m := BASELINE_RE.search(sentence)) else None),
    }


df_fwd = df_all[df_all["is_forward_commitment"]].copy()
parsed = df_fwd["sentence"].apply(parse_claim).apply(pd.Series)
df_parsed = pd.concat([df_fwd.reset_index(drop=True), parsed], axis=1)

df_parsed["fields_extracted"] = (
    df_parsed[["target_year", "target_value", "scope", "baseline"]]
    .notna()
    .sum(axis=1)
)

print("\n=== PARSEABILITY (forward-commitments only) ===")
print(df_parsed["fields_extracted"].value_counts().sort_index())
print(f"\nMean fields extracted per claim: "
      f"{df_parsed['fields_extracted'].mean():.2f} / 4")
print(f"Claims with >= 2 fields: "
      f"{(df_parsed['fields_extracted'] >= 2).sum()} "
      f"/ {len(df_parsed)} "
      f"({100 * (df_parsed['fields_extracted'] >= 2).mean():.1f}%)")

df_parsed.to_csv(os.path.join(output_path, "all_claims_v2_parsed.csv"), index=False)

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

MSFT: 1357 sentences -> 39 pre-filtered


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  -> classifier kept 33 as forward-commitment (saved 39 candidates total)
AAPL: 2339 sentences -> 69 pre-filtered
  -> classifier kept 57 as forward-commitment (saved 69 candidates total)
JPM: 1433 sentences -> 16 pre-filtered
  -> classifier kept 11 as forward-commitment (saved 16 candidates total)
WMT: 572 sentences -> 5 pre-filtered
  -> classifier kept 5 as forward-commitment (saved 5 candidates total)
CVX: 1479 sentences -> 16 pre-filtered
  -> classifier kept 16 as forward-commitment (saved 16 candidates total)
XOM: 890 sentences -> 2 pre-filtered
  -> classifier kept 2 as forward-commitment (saved 2 candidates total)
NEE: 1219 sentences -> 23 pre-filtered
  -> classifier kept 23 as forward-commitment (saved 23 candidates total)
PG: 279 sentences -> 6 pre-filtered
  -> classifier kept 6 as forward-commitment (saved 6 candidates total)

DONE. Total candidates across 8 companies: 176
Of these, classified as forward-commitment: 153

=== PARSEABILITY (forward-commitments only) ===
fi

### Cross Reference Code



In [ ]:
!pip install sentence-transformers scikit-learn beautifulsoup4 lxml

In [ ]:
import os, re
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Fast rule-based sentence splitter for the very large 10-K bodies
sent_nlp = spacy.blank("en")
sent_nlp.add_pipe("sentencizer")
sent_nlp.max_length = 5_000_000

embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Phase 2 setup ready.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Phase 2 setup ready.


In [ ]:
# Isolate the primary 10-K from the raw full-submission, strip HTML/XBRL, chunk clean text.
tenk_path = "/content/drive/My Drive/ESG_proj/10k_txt"
tickers   = ["MSFT", "AAPL", "JPM", "WMT", "CVX", "XOM", "NEE", "PG"]

DROP_TABLES = True   # 10-K financial tables are mostly numeric noise

def extract_primary_10k(submission_text: str) -> str:
    docs = re.findall(r"<DOCUMENT>(.*?)</DOCUMENT>", submission_text, flags=re.DOTALL)
    target = None
    for doc in docs:
        m = re.search(r"<TYPE>\s*([^\s<]+)", doc)
        if m and m.group(1).strip().upper() == "10-K":
            target = doc
            break
    if target is None:
        target = docs[0] if docs else submission_text

    m = re.search(r"<TEXT>(.*?)</TEXT>", target, flags=re.DOTALL)
    body = m.group(1) if m else target
    body = re.sub(r"</?(XBRL|XML)[^>]*>", " ", body, flags=re.IGNORECASE)

    soup = BeautifulSoup(body, "lxml")
    drop = ["script", "style"] + (["table"] if DROP_TABLES else [])
    for tag in soup(drop):
        tag.decompose()
    text = soup.get_text(separator="\n")

    text = re.sub(r"-\n", "", text)
    text = "\n".join(re.sub(r"[ \t\xa0]+", " ", ln).strip() for ln in text.split("\n"))
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()

def chunk_text(text, target_chars=400, max_chars=800, min_chars=40):
    doc = sent_nlp(text)
    sents = [s.text.strip() for s in doc.sents if s.text.strip()]
    chunks, buf, buflen = [], [], 0
    for s in sents:
        if len(s) > max_chars:
            for j in range(0, len(s), max_chars):
                chunks.append(s[j:j + max_chars])
            continue
        buf.append(s); buflen += len(s)
        if buflen >= target_chars:
            chunks.append(" ".join(buf)); buf, buflen = [], 0
    if buf:
        chunks.append(" ".join(buf))
    return [c for c in chunks if len(c) >= min_chars]

tenk_chunks = {}
for ticker in tickers:
    path = os.path.join(tenk_path, f"{ticker}_10K_2023.txt")
    if not os.path.exists(path):
        print(f"WARNING: {path} not found, skipping"); continue
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        raw = f.read()
    clean = extract_primary_10k(raw)
    chunks = chunk_text(clean)
    tenk_chunks[ticker] = chunks
    print(f"{ticker}: {len(clean):>9,} clean chars -> {len(chunks):>5,} chunks")

assert any(len(v) > 0 for v in tenk_chunks.values()), \
    "Still 0 chunks -- check that the .txt files are SEC full-submission text."

MSFT:   323,858 clean chars ->   601 chunks
AAPL:   179,255 clean chars ->   328 chunks
JPM: 1,116,479 clean chars -> 1,976 chunks
WMT:   332,023 clean chars ->   600 chunks
CVX:   406,394 clean chars ->   724 chunks
XOM:   336,295 clean chars ->   603 chunks
NEE:   512,839 clean chars ->   905 chunks
PG:   273,574 clean chars ->   505 chunks


In [ ]:
claims_path = "/content/drive/My Drive/ESG_proj/extracted_claims_v2/all_claims_v2_parsed.csv"
df_claims = pd.read_csv(claims_path)
df_fwd = (df_claims[df_claims["is_forward_commitment"] == True]
          .copy().reset_index(drop=True))
print(f"Embedding {len(df_fwd)} forward-commitment claims...")
claim_embeddings = embedder.encode(df_fwd["sentence"].tolist(), show_progress_bar=True)

tenk_embeddings = {}
for ticker, chunks in tenk_chunks.items():
    if not chunks:
        print(f"  {ticker}: 0 chunks, skipping"); continue
    tenk_embeddings[ticker] = embedder.encode(chunks, show_progress_bar=False)
    print(f"  {ticker}: embedded {len(chunks)} chunks")



Embedding 153 forward-commitment claims...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

  MSFT: embedded 601 chunks
  AAPL: embedded 328 chunks
  JPM: embedded 1976 chunks
  WMT: embedded 600 chunks
  CVX: embedded 724 chunks
  XOM: embedded 603 chunks
  NEE: embedded 905 chunks
  PG: embedded 505 chunks


In [ ]:
TOP_K = 3
SIM_THRESHOLD = 0.35   # below -> 10-K doesn't discuss the topic ("calling - silent")

PCT_RE   = re.compile(r"(\d+(?:\.\d+)?)\s*(?:percent|%)", re.IGNORECASE)
SCOPE_RE = re.compile(r"\bscope\s*([123])\b", re.IGNORECASE)

EMIT_RE  = re.compile(r"\b(emission|carbon|ghg|greenhouse|scope\s*[123]|decarboni|renewable)",
                      re.IGNORECASE)
REDUCE_RE = re.compile(r"\b(reduc|cut|lower|decreas|abat|declin|down)\w*", re.IGNORECASE)


def pct_near_emissions(text, window=70, require_reduction=True):
    for m in PCT_RE.finditer(text):
        ctx = text[max(0, m.start() - window): m.end() + window]
        if EMIT_RE.search(ctx) and (not require_reduction or REDUCE_RE.search(ctx)):
            return float(m.group(1))
    return None

def scopes_in(text):
    return set(SCOPE_RE.findall(text))

def classify(best_sim, claim_pct, match_pct, claim_scopes, match_scopes):
    if best_sim < SIM_THRESHOLD:
        return "silent", np.nan
    scope_ok = (not claim_scopes) or (not match_scopes) or bool(claim_scopes & match_scopes)
    if claim_pct is not None and match_pct is not None and scope_ok:
        return "quantified", round(match_pct - claim_pct, 1)   # negative => under-delivered
    return "addressed_unquantified", np.nan

rows = []
for idx, claim_row in df_fwd.iterrows():
    ticker = claim_row["ticker"]
    if ticker not in tenk_embeddings:
        continue
    sims = cosine_similarity(claim_embeddings[idx].reshape(1, -1),
                             tenk_embeddings[ticker])[0]
    order = np.argsort(sims)[::-1][:TOP_K]
    best_idx = int(order[0]); best_sim = float(sims[best_idx])
    best_chunk = tenk_chunks[ticker][best_idx]

    claim_pct = pct_near_emissions(claim_row["sentence"])
    match_pct = pct_near_emissions(best_chunk)
    verdict, gap = classify(best_sim, claim_pct, match_pct,
                            scopes_in(claim_row["sentence"]), scopes_in(best_chunk))

    rows.append({
        "ticker": ticker, "claim": claim_row["sentence"],
        "claim_pct": claim_pct, "target_year": claim_row.get("target_year"),
        "scope": claim_row.get("scope"),
        "best_match": best_chunk, "match_pct": match_pct,
        "cos_sim": round(best_sim, 3),
        "quant_gap": gap,
        "verdict": verdict,
    })

df_xref = pd.DataFrame(rows)
output_path = "/content/drive/My Drive/ESG_proj/crossref_results"
os.makedirs(output_path, exist_ok=True)
df_xref.to_csv(os.path.join(output_path, "crossref_results.csv"), index=False)
print(f"Cross-reference complete: {len(df_xref)} claim-outcome pairs")
print(df_xref["verdict"].value_counts())

Cross-reference complete: 153 claim-outcome pairs
verdict
addressed_unquantified    145
silent                      8
Name: count, dtype: int64


In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 70)

print("=== similarity distribution ===")
print(df_xref["cos_sim"].describe().round(3))

print("\n=== the 3 quantified (the real comparisons) ===")
print(df_xref[df_xref.verdict=="quantified"][["ticker","claim","claim_pct","match_pct","quant_gap"]].to_string(index=False))

print("\n=== the 8 silent (lowest-sim, supposedly off-topic) ===")
print(df_xref[df_xref.verdict=="silent"][["ticker","cos_sim","claim"]].to_string(index=False))

=== similarity distribution ===
count    153.000
mean       0.528
std        0.108
min        0.293
25%        0.452
50%        0.515
75%        0.600
max        0.840
Name: cos_sim, dtype: float64

=== the 3 quantified (the real comparisons) ===
Empty DataFrame
Columns: [ticker, claim, claim_pct, match_pct, quant_gap]
Index: []

=== the 8 silent (lowest-sim, supposedly off-topic) ===
ticker  cos_sim                                                                                                                                                                                                                                                                                                                claim
  AAPL    0.307                                                                                                                                                   And to accelerate collective efforts, we signed on to the First Movers Coalition’s near-zero emissions primary aluminum commi

In [ ]:
# (1) Linguistic divergence: did they stop addressing the claim?
df_xref["linguistic_divergence"] = (1.0 - df_xref["cos_sim"]).clip(0, 1)

# (2) Quantitative divergence: only where a comparable emissions number exists on both sides.
def shortfall(row):
    if row["verdict"] != "quantified" or pd.isna(row["quant_gap"]):
        return np.nan
    return max(0.0, -row["quant_gap"])
df_xref["quant_shortfall"] = df_xref.apply(shortfall, axis=1)

summary = (
    df_xref.groupby("ticker")
    .agg(
        total_claims        =("claim", "count"),
        silent              =("verdict", lambda x: (x == "silent").sum()),
        addressed           =("verdict", lambda x: (x == "addressed_unquantified").sum()),
        quantified          =("verdict", lambda x: (x == "quantified").sum()),
        silence_rate        =("verdict", lambda x: round((x == "silent").mean(), 3)),
        mean_linguistic_div =("linguistic_divergence", "mean"),
        mean_cos_sim        =("cos_sim", "mean"),
        mean_quant_shortfall=("quant_shortfall", "mean"),
    )
    .round(3)
)
print("DIVERGENCE SUMMARY (two components, separate)")
print(summary)
summary.to_csv(os.path.join(output_path, "divergence_summary.csv"))

DIVERGENCE SUMMARY (two components, separate)
        total_claims  silent  addressed  quantified  silence_rate  \
ticker                                                              
AAPL              57       8         49           0          0.14   
CVX               16       0         16           0          0.00   
JPM               11       0         11           0          0.00   
MSFT              33       0         33           0          0.00   
NEE               23       0         23           0          0.00   
PG                 6       0          6           0          0.00   
WMT                5       0          5           0          0.00   
XOM                2       0          2           0          0.00   

        mean_linguistic_div  mean_cos_sim  mean_quant_shortfall  
ticker                                                           
AAPL                  0.539         0.461                   NaN  
CVX                   0.356         0.644                   NaN  

In [ ]:
'''# saving annotation sample - one time use only, leaving the code in
parts = []
for tkr, g in df_xref.groupby("ticker"):
    parts.append(g.sample(min(4, len(g)), random_state=42))
sample = pd.concat(parts).reset_index(drop=True)

# two independent passes + a reconciled column Block 7 will read
sample["label_you"]      = ""
sample["label_teammate"] = ""
sample["human_label"]    = ""
sample["notes"]          = ""

sample_path = os.path.join(output_path, "manual_annotation_sample.csv")
sample.to_csv(sample_path, index=False)
print(f"Annotation sample saved: {len(sample)} rows -> {sample_path}")
print("1) Each person fills their own column independently.")
print("2) Compare, reconcile disagreements into 'human_label'.")
print("3) Then run the next cell.")'''

Annotation sample saved: 30 rows -> /content/drive/My Drive/ESG_proj/crossref_results/manual_annotation_sample.csv
1) Each person fills their own column independently.
2) Compare, reconcile disagreements into 'human_label'.
3) Then run the next cell.


In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import cohen_kappa_score
from scipy.stats import spearmanr

df_ann = pd.read_csv(os.path.join(output_path, "manual_annotation_sample.csv"))

# normalize labels: lowercase, strip, map synonyms, drop blanks
SYN = {"confirmed":"fulfilled","fulfilled":"fulfilled","partial":"partial",
       "unfulfilled":"unfulfilled","silent":"silent"}
def norm(col):
    return df_ann[col].astype(str).str.strip().str.lower().map(SYN)

df_ann["human_label"] = norm("human_label")
df_h = df_ann[df_ann["human_label"].notna()].copy()
print(f"Clean labeled rows: {len(df_h)}")
print(df_h["human_label"].value_counts())

# inter-annotator agreement on the clean set
yl, tl = norm("label_you"), norm("label_teammate")
mask = yl.notna() & tl.notna()
if mask.sum():
    print(f"\nInter-annotator: {(yl[mask]==tl[mask]).mean():.0%} agree, "
          f"kappa={cohen_kappa_score(yl[mask], tl[mask]):.3f}")

# the key tuning loop: silence detection (silent vs addressed)
# The pipeline can only decide silent-vs-not (no quantified pairs exist),
# so we tune the threshold to best match human 'silent' calls.
df_h["human_silent"] = df_h["human_label"].eq("silent")
print("\nthresh  acc   kappa  n_pred_silent")
for thr in [round(x,2) for x in np.arange(0.35, 0.70, 0.05)]:
    pred_silent = df_h["cos_sim"] < thr
    acc = (pred_silent == df_h["human_silent"]).mean()
    k = cohen_kappa_score(df_h["human_silent"], pred_silent)
    print(f"{thr:.2f}   {acc:.2f}  {k:+.3f}   {int(pred_silent.sum())}")

# does the continuous score track human judgment?
order = {"fulfilled":0,"partial":1,"unfulfilled":2,"silent":3}
rho, p = spearmanr(df_h["linguistic_divergence"], df_h["human_label"].map(order),
                   nan_policy="omit")
print(f"\nSpearman(divergence, human ordinal) = {rho:.3f} (p={p:.3f})")

Clean labeled rows: 30
human_label
silent       17
partial       9
fulfilled     4
Name: count, dtype: int64

Inter-annotator: 87% agree, kappa=0.768

thresh  acc   kappa  n_pred_silent
0.35   0.43  +0.000   0
0.40   0.53  +0.157   3
0.45   0.60  +0.265   5
0.50   0.63  +0.298   10
0.55   0.70  +0.405   14
0.60   0.63  +0.233   20
0.65   0.70  +0.348   24

Spearman(divergence, human ordinal) = 0.501 (p=0.005)


**Threshold calibration:** The initial SIM_THRESHOLD = 0.35 was a placeholder. Against 30 hand-labeled claim-disclosure pairs (inter-annotator kappa = 0.77), we swept thresholds and selected 0.55, which maximized agreement with human "silent" judgments (kappa = 0.41). We re-run the cross-reference and divergence summary below with the calibrated value.

In [ ]:
# rerunning with the right threshold = 0.55
TOP_K = 3
SIM_THRESHOLD = 0.55   # below -> 10-K doesn't discuss the topic ("silent")

PCT_RE   = re.compile(r"(\d+(?:\.\d+)?)\s*(?:percent|%)", re.IGNORECASE)
SCOPE_RE = re.compile(r"\bscope\s*([123])\b", re.IGNORECASE)

EMIT_RE  = re.compile(r"\b(emission|carbon|ghg|greenhouse|scope\s*[123]|decarboni|renewable)",
                      re.IGNORECASE)
REDUCE_RE = re.compile(r"\b(reduc|cut|lower|decreas|abat|declin|down)\w*", re.IGNORECASE)


def pct_near_emissions(text, window=70, require_reduction=True):
    for m in PCT_RE.finditer(text):
        ctx = text[max(0, m.start() - window): m.end() + window]
        if EMIT_RE.search(ctx) and (not require_reduction or REDUCE_RE.search(ctx)):
            return float(m.group(1))
    return None

def scopes_in(text):
    return set(SCOPE_RE.findall(text))

def classify(best_sim, claim_pct, match_pct, claim_scopes, match_scopes):
    if best_sim < SIM_THRESHOLD:
        return "silent", np.nan
    scope_ok = (not claim_scopes) or (not match_scopes) or bool(claim_scopes & match_scopes)
    if claim_pct is not None and match_pct is not None and scope_ok:
        return "quantified", round(match_pct - claim_pct, 1)   # negative => under-delivered
    return "addressed_unquantified", np.nan

rows = []
for idx, claim_row in df_fwd.iterrows():
    ticker = claim_row["ticker"]
    if ticker not in tenk_embeddings:
        continue
    sims = cosine_similarity(claim_embeddings[idx].reshape(1, -1),
                             tenk_embeddings[ticker])[0]
    order = np.argsort(sims)[::-1][:TOP_K]
    best_idx = int(order[0]); best_sim = float(sims[best_idx])
    best_chunk = tenk_chunks[ticker][best_idx]

    claim_pct = pct_near_emissions(claim_row["sentence"])
    match_pct = pct_near_emissions(best_chunk)
    verdict, gap = classify(best_sim, claim_pct, match_pct,
                            scopes_in(claim_row["sentence"]), scopes_in(best_chunk))

    rows.append({
        "ticker": ticker, "claim": claim_row["sentence"],
        "claim_pct": claim_pct, "target_year": claim_row.get("target_year"),
        "scope": claim_row.get("scope"),
        "best_match": best_chunk, "match_pct": match_pct,
        "cos_sim": round(best_sim, 3),
        "quant_gap": gap,
        "verdict": verdict,
    })

df_xref = pd.DataFrame(rows)
output_path = "/content/drive/My Drive/ESG_proj/crossref_results"
os.makedirs(output_path, exist_ok=True)
df_xref.to_csv(os.path.join(output_path, "crossref_results.csv"), index=False)
print(f"Cross-reference complete: {len(df_xref)} claim-outcome pairs")
print(df_xref["verdict"].value_counts())

Cross-reference complete: 153 claim-outcome pairs
verdict
silent                    89
addressed_unquantified    64
Name: count, dtype: int64


In [ ]:
# (1) Linguistic divergence: did they stop addressing the claim?
df_xref["linguistic_divergence"] = (1.0 - df_xref["cos_sim"]).clip(0, 1)

# (2) Quantitative divergence: only where a comparable emissions number exists on both sides.
def shortfall(row):
    if row["verdict"] != "quantified" or pd.isna(row["quant_gap"]):
        return np.nan
    return max(0.0, -row["quant_gap"])
df_xref["quant_shortfall"] = df_xref.apply(shortfall, axis=1)

summary = (
    df_xref.groupby("ticker")
    .agg(
        total_claims        =("claim", "count"),
        silent              =("verdict", lambda x: (x == "silent").sum()),
        addressed           =("verdict", lambda x: (x == "addressed_unquantified").sum()),
        quantified          =("verdict", lambda x: (x == "quantified").sum()),
        silence_rate        =("verdict", lambda x: round((x == "silent").mean(), 3)),
        mean_linguistic_div =("linguistic_divergence", "mean"),
        mean_cos_sim        =("cos_sim", "mean"),
        mean_quant_shortfall=("quant_shortfall", "mean"),
    )
    .round(3)
)
print("=== DIVERGENCE SUMMARY (two components, separate) ===")
print(summary)
summary.to_csv(os.path.join(output_path, "divergence_summary.csv"))

=== DIVERGENCE SUMMARY (two components, separate) ===
        total_claims  silent  addressed  quantified  silence_rate  \
ticker                                                              
AAPL              57      46         11           0         0.807   
CVX               16       2         14           0         0.125   
JPM               11      11          0           0         1.000   
MSFT              33      12         21           0         0.364   
NEE               23      14          9           0         0.609   
PG                 6       3          3           0         0.500   
WMT                5       1          4           0         0.200   
XOM                2       0          2           0         0.000   

        mean_linguistic_div  mean_cos_sim  mean_quant_shortfall  
ticker                                                           
AAPL                  0.539         0.461                   NaN  
CVX                   0.356         0.644                